# Water Quality Index Estimation

This notebook uses water quality data to calculate the Water Quality Index and predict water quality using machine learning.

## Step 1: Upload the dataset

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

if not os.path.exists("water_dataX.csv"):
    from google.colab import files
    uploaded = files.upload()

## Step 2: Read the data

In [ ]:
data = pd.read_csv("water_dataX.csv", encoding="latin-1")
data.head()

## Step 3: Clean the data

In [ ]:
data.columns = ["code", "location", "state", "temp", "do", "ph", "cond", "bod", "nitrate", "fc", "tc", "year"]

columns = ["temp", "do", "ph", "cond", "bod", "nitrate", "fc", "tc"]

for col in columns:
    data[col] = pd.to_numeric(data[col], errors="coerce")

data = data.drop_duplicates()
data.loc[(data["ph"] < 0) | (data["ph"] > 14), "ph"] = np.nan

print("Rows and columns:", data.shape)
print(data.isnull().sum())

## Step 4: Calculate the Water Quality Index

In [ ]:
standards = {
    "ph": 8.5,
    "do": 5,
    "bod": 3,
    "cond": 2250,
    "nitrate": 10,
    "fc": 500,
    "tc": 500
}

weights = {}
for col in standards:
    weights[col] = 1 / standards[col]

total = sum(weights.values())

for col in weights:
    weights[col] = weights[col] / total

water = data.dropna(subset=list(standards.keys())).copy()

quality = pd.DataFrame(index=water.index)
quality["ph"] = 100 * abs(water["ph"] - 7) / (8.5 - 7)
quality["do"] = 100 * (14.6 - water["do"]) / (14.6 - 5)

for col in ["bod", "cond", "nitrate", "fc", "tc"]:
    quality[col] = 100 * water[col] / standards[col]

quality = quality.clip(0, 100)

water["wqi"] = 0
for col in standards:
    water["wqi"] = water["wqi"] + quality[col] * weights[col]

water["category"] = np.where(water["wqi"] <= 50, "Good", np.where(water["wqi"] <= 75, "Poor", "Very Poor"))

water[["year", "wqi", "category"]].head()

## Step 5: Explore the WQI values

In [ ]:
print(water["wqi"].describe())
print(water["category"].value_counts())

water["wqi"].hist(bins=20)
plt.xlabel("Water Quality Index")
plt.ylabel("Number of records")
plt.title("Distribution of WQI")
plt.show()

## Step 6: Prepare the data for machine learning

In [ ]:
features = ["temp", "do", "ph", "cond", "bod", "nitrate", "fc", "tc"]

water = water.sort_values("year")

train = water[water["year"] <= 2012]
test = water[water["year"] >= 2013]

X_train = train[features]
X_test = test[features]
y_train = train["wqi"]
y_test = test["wqi"]

print("Training records:", len(train))
print("Testing records:", len(test))

## Step 7: Train regression models

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

models = {
    "Linear Regression": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LinearRegression()),
    "Decision Tree": make_pipeline(SimpleImputer(strategy="median"), DecisionTreeRegressor(random_state=42)),
    "Random Forest": make_pipeline(SimpleImputer(strategy="median"), RandomForestRegressor(n_estimators=100, random_state=42))
}

regression_results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    regression_results[name] = [mae, rmse, r2]
    print(name)
    print("MAE:", round(mae, 2))
    print("RMSE:", round(rmse, 2))
    print("R2 Score:", round(r2, 2))
    print()

## Step 8: Compare regression models

In [ ]:
results = pd.DataFrame(regression_results, index=["MAE", "RMSE", "R2 Score"]).T
results

## Step 9: Predict water quality category

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

encoder = LabelEncoder()
water["category_number"] = encoder.fit_transform(water["category"])

train = water[water["year"] <= 2012]
test = water[water["year"] >= 2013]

X_train = train[features]
X_test = test[features]
y_train = train["category_number"]
y_test = test["category_number"]

classifiers = {
    "Logistic Regression": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000)),
    "Decision Tree": make_pipeline(SimpleImputer(strategy="median"), DecisionTreeClassifier(random_state=42)),
    "Random Forest": make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=100, random_state=42))
}

for name, model in classifiers.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    print(name)
    print("Accuracy:", round(accuracy_score(y_test, predictions), 2))
    print(classification_report(y_test, predictions, target_names=encoder.classes_, zero_division=0))

## Step 10: Check which features are important

In [ ]:
forest = models["Random Forest"]
forest_model = forest.named_steps["randomforestregressor"]

importance = pd.DataFrame({
    "Feature": features,
    "Importance": forest_model.feature_importances_
})

importance = importance.sort_values("Importance", ascending=False)
print(importance)

plt.bar(importance["Feature"], importance["Importance"])
plt.xticks(rotation=45)
plt.xlabel("Water quality parameters")
plt.ylabel("Importance")
plt.title("Feature Importance")
plt.tight_layout()
plt.show()

## Step 11: Actual and predicted WQI

In [ ]:
best_model = models["Random Forest"]
predictions = best_model.predict(X_test)

plt.scatter(y_test, predictions)
plt.xlabel("Actual WQI")
plt.ylabel("Predicted WQI")
plt.title("Actual vs Predicted WQI")
plt.show()

## Conclusion

The notebook calculates the Water Quality Index from the available water quality parameters. It then uses regression models to estimate WQI values and classification models to predict the water quality category. The model scores can be used to compare how well the different machine learning models perform.